In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_34.pickle

In [ ]:
%%RecordEvent
%%time
### cell 35 ###

### cell 35 optimized for cudf ###

# --- align 2018 columns to the new question text ---
question_of_interest_old_cell47 = (
    'What machine learning frameworks have you used in the past 5 years?'
)
question_of_interest_cell47 = (
    'Which of the following machine learning frameworks do you use on a regular basis?'
)
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace(
            question_of_interest_old_cell47,
            question_of_interest_cell47,
            regex=False
        )
)

# --- optimized subset grab (uses GPU-friendly vectorized ops) ---
def grab_subset_of_data_47(df, question):
    # select columns via vectorized contains
    cols = df.columns[df.columns.str.contains(question)]
    subset = df[cols]
    # vectorized rename: split on last '-', strip leading spaces
    new_cols = cols.str.rsplit('-', 1).str.get(-1).str.lstrip()
    subset.columns = new_cols
    # drop rows where all responses are null
    return subset.dropna(how='all')

# --- combine by year (unchanged logic) ---
def combine_subsets(question, include_2017=False):
    year_dfs = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017:
        year_dfs.insert(0, ('2017', responses_df_2017))
    subsets = [grab_subset_of_data_47(df, question) for _, df in year_dfs]
    years = [yr for yr, _ in year_dfs]
    df_combined = pd.concat(subsets, keys=years, names=['year'])
    df_combined = df_combined.reset_index(level='year').reset_index(drop=True)
    df_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_counts

# --- convert counts to percentages (GPU-friendly arithmetic) ---
def convert_df_of_counts_to_percentages_47(df, df_counts):
    denom = df['year'].value_counts().sort_index()
    pct = df_counts.set_index('year').div(denom, axis=0) * 100
    return pct.reset_index()

# --- main pipeline ---
ml_df_combined_cell47, ml_df_combined_counts_cell47 = combine_subsets(
    question_of_interest_cell47,
    include_2017=False
)

# copy and bulk-combine framework columns
ml_df_combined_2_cell47 = ml_df_combined_cell47.copy()
combos = [
    (['TensorFlow', 'TensorFlow ', 'Keras', 'Keras '],
     'TensorFlow/Keras', 'TensorFlow/Keras'),
    (['PyTorch', 'PyTorch ', 'PyTorch Lightning ', 'Fast.ai ', 'Fastai'],
     'PyTorch/Lightning/Fast.ai', 'PyTorch/PyTorch Lightning/Fast.ai'),
    (['Xgboost', 'Xgboost ', 'lightgbm', 'LightGBM ', 'catboost', 'CatBoost '],
     'Xgboost/LightGBM/CatBoost', 'Xgboost/LightGBM/CatBoost'),
    (['Scikit—learn ', 'Scikit—learn ', 'learn ', 'Learn'],
     'Scikit-learn', 'Scikit-learn')
]
# gather all original columns for a single drop
orig_cols = [c for cols, _, _ in combos for c in cols]

for cols, new_col, label in combos:
    existing = [c for c in cols if c in ml_df_combined_2_cell47.columns]
    if existing:
        mask = ml_df_combined_2_cell47[existing].notna().any(axis=1)
        ml_df_combined_2_cell47[new_col] = mask.map({True: label, False: None})
    else:
        # create empty column if none of the combo columns exist
        ml_df_combined_2_cell47[new_col] = None

# drop all original individual framework columns in one go
ml_df_combined_2_cell47 = ml_df_combined_2_cell47.drop(
    columns=orig_cols,
    errors='ignore'
)

# recalc counts & percentages
ml_df_combined_counts_2_cell47 = (
    ml_df_combined_2_cell47.groupby('year').count().reset_index()
)
ml_df_combined_percentages_cell47 = convert_df_of_counts_to_percentages_47(
    ml_df_combined_2_cell47,
    ml_df_combined_counts_2_cell47
)

# select & order
ml_df_combined_percentages_cell47 = ml_df_combined_percentages_cell47[
    ['year', 'Scikit-learn', 'TensorFlow/Keras',
     'PyTorch/Lightning/Fast.ai', 'Xgboost/LightGBM/CatBoost']
]

# melt & sort for final output
df_cell47 = (
    ml_df_combined_percentages_cell47
    .melt(
        id_vars=['year'],
        value_vars=[
            'Scikit-learn', 'Xgboost/LightGBM/CatBoost',
            'TensorFlow/Keras', 'PyTorch/Lightning/Fast.ai'
        ],
        var_name='',
        value_name='value'
    )
    .sort_values(['year', 'value'])
    .reset_index(drop=True)
)

# display result
df_cell47

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_35_try_3.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_35_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[35], f)


In [ ]:
opt_output = Out.get(4)